In [2]:
import shutil
import os

# unzipped refined_verified_labels_for_yolo_segmentation.zip

# shutil.unpack_archive("refined_verified_labels_for_yolo_segmentation.zip", "refined_verified_labels_for_yolo_segmentation")


print(len(os.listdir("refined_verified_labels_for_yolo_segmentation")))

# unzipped tiles_jpg_final_zip.zip
#shutil.unpack_archive("tiles_jpg_final_zip.zip" ,"tiles_jpg_final_unzipped")

print(len(os.listdir("tiles_jpg_final_unzipped")))

1024
1600


In [3]:
label_dir = "refined_verified_labels_for_yolo_segmentation"
image_dir = "tiles_jpg_final_unzipped"
base = "/home3/zpdv81/YOLO_Segmentation_dataset_refined_label"

# Create YOLO folder structure

for split in ["train","val"]:
    os.makedirs(os.path.join(base,"images",split) , exist_ok = True)
    os.makedirs(os.path.join(base,"labels",split) , exist_ok = True)
    
    
# width of original raster image
raster_width = 20000
val_cutoff = raster_width*0.8 # 80% train  , 20% validation


# assign a tile to train or validation based on its left x-coordinate
def get_split(tile_name):
    parts = tile_name.split("_")
    left = int(parts[2])
    if left < val_cutoff:
        return "train"
    else:
        return "val"

train_count = 0
val_count = 0

# copy each image and its matching yolo label file

for label_file in os.listdir(label_dir):
    
    if not label_file.endswith(".txt"):  # only process yolo text label files
        continue
        
    # remove .txt to get matching image file name
    tile_name = os.path.splitext(label_file)[0]
    image_path = os.path.join(image_dir , tile_name + ".jpg")
    
    # skip labels without images matching
    if not os.path.exists(image_path):
        print(f" Missing image for : {label_file}")
        continue
    
    # check whether this tile belongs in train or validation
    split = get_split(tile_name)
    
    # source paths
    label_path = os.path.join(label_dir , label_file)
    
    
    # path destination
    output_label_path = os.path.join(base,'labels',split,label_file)
    output_image_path = os.path.join(base,'images' , split , tile_name + ".jpg")
    
    # copy files
    shutil.copy(label_path , output_label_path)
    shutil.copy(image_path , output_image_path)
    
    if split == "train":
        train_count +=1
    else:
        val_count +=1
    
    
print(f" Train images : {train_count}")
print(f" Validation images : {val_count}")

total = train_count + val_count

print(f" Train percentage : {train_count / total * 100 :.2f}%")
print(f" Validation Percentage : {val_count / total * 100 :.2f}%")


# create YOLO data.yaml file
data_yaml_content = f"""train : {base}/images/train
val: {base}/images/val

nc: 1
names:
 0: ridge_and_furrow
"""

yaml_path = os.path.join(base,"data.yaml")

with open(yaml_path,"w") as file:
    file.write(data_yaml_content)
    
print(f"data.yaml saved to : {yaml_path}")


 Train images : 894
 Validation images : 130
 Train percentage : 87.30%
 Validation Percentage : 12.70%
data.yaml saved to : /home3/zpdv81/YOLO_Segmentation_dataset_refined_label/data.yaml


In [5]:
training_script = f'''
from ultralytics import YOLO

# load pre trained yolo segment model

model = YOLO("yolov8n-seg.pt")

#train the model
results = model.train(
        data = "{base}/data.yaml",
        epochs = 10,
        imgsz = 640,
        batch = 16,
        device = 0,
        workers = 2,
        scale = 0.9,
        project = "/home3/zpdv81/yolo_runs_refined",
        name = "rnf_refined_yolo_segmentation_sanity_check",
        )
        '''
# save the training script
with open("/home3/zpdv81/train_confirm.py","w")as f:
    f.write(training_script)
print("Script written")

Script written


In [6]:
!/home3/zpdv81/yolo_venv/bin/python /home3/zpdv81/train_confirm.py

New https://pypi.org/project/ultralytics/8.4.118 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.110 🚀 Python-3.8.10 torch-2.4.1+cu121 CUDA:0 (NVIDIA A100 80GB PCIe MIG 1g.10gb, 9728MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home3/zpdv81/YOLO_Segmentation_dataset_refined_label/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, l

In [7]:
training_script = f'''
from ultralytics import YOLO

# load pre trained yolo segment model

model = YOLO("yolov8n-seg.pt")

#train the model
results = model.train(
        data = "{base}/data.yaml",
        epochs = 100,
        imgsz = 640,
        batch = 16,
        device = 0,
        workers = 2,
        scale = 0.9,
        project = "/home3/zpdv81/yolo_runs_refined",
        name = "rnf_refined_yolo_segmentation_for_project",
        )
        '''
# save the training script
with open("/home3/zpdv81/train_confirm.py","w")as f:
    f.write(training_script)
print("Script written")

Script written


In [8]:
!/home3/zpdv81/yolo_venv/bin/python /home3/zpdv81/train_confirm.py

New https://pypi.org/project/ultralytics/8.4.118 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.110 🚀 Python-3.8.10 torch-2.4.1+cu121 CUDA:0 (NVIDIA A100 80GB PCIe MIG 1g.10gb, 9728MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home3/zpdv81/YOLO_Segmentation_dataset_refined_label/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, 